In [ ]:
import os
import SimpleITK as sitk
import numpy as np
import glob

axial_harmo1 = r'/path/to/workspace/classificationmodel_harmonizednew/sagittal_harmonize/test/'
# axial_harmo2 = r'/path/to/workspace/classificationmodel_harmonizednew/axial_harmonize/val/'
image_path1 = glob.glob(os.path.join(axial_harmo1, '*.nii.gz'))
# image_path2 = glob.glob(os.path.join(axial_harmo2, '*.nii.gz'))
image_path = image_path1 
print(image_path)
print(len(image_path))

In [ ]:

def compute_rectum_center(colon_mask, rectum_z_length, colon_y_length):

    # colon_voxels will be a tuple of (z_coords, y_coords, x_coords)
    z_coords, y_coords, x_coords = np.where(colon_mask > 0)
    
    # Find min/max along each axis for the colon region
    z_min, z_max = z_coords.min(), z_coords.max()
    y_min, y_max = y_coords.min(), y_coords.max()
    x_min, x_max = x_coords.min(), x_coords.max()
    
    print("z_min", z_min)
    print("z_max", z_max)
    print("y_min", y_min)
    print("y_max", y_max)
    print("x_min", x_min)
    print("x_max", x_max)
    
    x_center = (x_min + x_max) // 2
    # y_center = (y_min + y_max) // 2
    y_center = y_min + colon_y_length // 2
    z_center =  z_min - rectum_z_length // 2

    center = (x_center, y_center, z_center)
    
    return center

In [ ]:
def center_crop1(image, center, crop_size):
    """
    Perform center cropping on the original image based on the center point.
    :param image: SimpleITK Image object.
    :param center: (x, y, z) coordinates of the center point.
    :param crop_size: (width, height, depth) cropping size.
    :return: Cropped SimpleITK Image.
    """
    # Get image size (x, y, z)
    size = image.GetSize()
    spacing = image.GetSpacing()

    # Calculate start position for cropping
    start = [max(0, min(int(center[i]) - int(crop_size[i]) // 2, int(size[i]) - int(crop_size[i]))) for i in range(3)]

    # Adjust cropping size if it exceeds image dimensions
    adjusted_size = [min(int(crop_size[i]), int(size[i]) - int(start[i])) for i in range(3)]

    # Use SimpleITK RegionOfInterest to crop the image
    cropped_image = sitk.RegionOfInterest(image, adjusted_size, start)

    # Adjust spatial information for the cropped image
    new_origin = [image.GetOrigin()[i] + start[i] * spacing[i] for i in range(3)]
    cropped_image.SetOrigin(new_origin)
    cropped_image.SetSpacing(spacing)
    cropped_image.SetDirection(image.GetDirection())

    return cropped_image


In [ ]:
# Load the original MR image
path = r'/path/to/workspace/classificationmodel_original_sagittal/total_work/resampled'
new_dir = r'/path/to/workspace/classificationmodel_harmonizednew/sagittal_harmonize/rectum_centrecrop/test'

crop_size = (48, 192, 192)
rectum_z_length = 80  #Approximate estimation of rectum pixel length on image
colon_y_length = 120

for img_path in image_path:
    img = sitk.ReadImage(img_path)
    patient_id = os.path.basename(img_path).split('_')[1]

    search_pattern = os.path.join(path, f"{patient_id}*.nii.gz")
    matching_patches = glob.glob(search_pattern)

    if matching_patches:
        print(f"found patches for patient {patient_id}:")
        for patch_path in matching_patches:
            print(patch_path)
            colon_path =  os.path.join(patch_path, 'colon.nii.gz')
            colon_img = sitk.ReadImage(colon_path)
            colon_mask = sitk.GetArrayFromImage(colon_img)
          
            try:
                # Calculate the center point of the patch
                center = compute_rectum_center(colon_mask, rectum_z_length, colon_y_length)
                print(f"Center point of the patch: {center}")

                # Perform center cropping on the original image
                cropped_img = center_crop1(img, center, crop_size)

                # Save the cropped image
                file_name = patient_id + '.nii.gz' #patient number
                output_path = os.path.join(new_dir, file_name)

                sitk.WriteImage(cropped_img, output_path)
                print(f"Cropped image saved to {output_path}")
            
            except Exception as e:
                print(f"An error occurred: {e}")
                print("patient number:", patient_id)
            
    else:
        print(f"no patches found for patient {patient_id}")
            
print('done')